# Занятие 12. Стерео: от эпиполярных линий к глубине

За 90 минут: **ректификация → SGBM/WLS → проверка по GT → глубина и облако**.
Три TODO по 5–12 строк; остальной конвейер готов. Понадобится окружение курса
с `opencv-contrib-python==4.14.0.94`. После установки работа полностью офлайн,
камера и внешние датасеты не нужны. Запускайте ячейки сверху вниз.

Опора на L12: **страницы PDF** 8–16 (геометрия), 28–32 и 49–52 (ректификация),
33–39 (триангуляция и точность), 53 (SGBM). План занятия: 15 / 50 / 15 / 10 минут.
WLS и количественная проверка развивают конвейер лекции по плану курса.

Используем свою синтетическую сцену: две текстурированные плоскости на Z=2 и 4 m,
камеры с базой B=0.12 m и поворотом правой камеры. **GT (ground truth)** вычисляется
из геометрии в сетке ректифицированного левого изображения; это контроль,
он не подаётся в matcher. Числа этого набора не являются результатами Middlebury/KITTI.
Генератор и SHA-256 — в `data/README.md`. Готовый результат перейдёт в стереочасть HW4.

In [ ]:
import importlib.util
import itertools
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

assert cv2.__version__ == '4.14.0', 'Выберите ядро окружения курса .venv'
if 'get_ipython' in globals():
    get_ipython().run_line_magic('matplotlib', 'inline')
if 'LAB_DIR' not in globals():
    LAB_DIR = next(p for p in (Path.cwd(), Path.cwd() / 'labs/lab12-stereo')
                   if (p / 'make.py').is_file())
spec = importlib.util.spec_from_file_location('lab12_data', LAB_DIR / 'make.py')
data_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_module)
make_scene, array_hashes = data_module.make_scene, data_module.array_hashes

## 0–15 минут. Геометрия до вычисления диспаратности

Система координат мира совпадает с исходной левой камерой.
`X_right = R @ X_left + T`; центр правой камеры `C_right = -R.T @ T`.
Поэтому `T` здесь имеет отрицательную X-компоненту, хотя длина базы положительна.
Пиксели обозначаем `(u,v)`, размер изображения — `(W,H)`.

Для однородных пикселей: `x_right.T @ F @ x_left = 0`,
`E = [T]_x @ R`, `F = inv(K_right).T @ E @ inv(K_left)`.
`computeCorrespondEpilines(..., whichImage=1, F)` даёт линии **на правом** кадре.
Расстояние до линии в пикселях — `abs(a*u+b*v+c)/hypot(a,b)`.
Не оцениваем F из идеальных соответствий: здесь известная калибровка позволяет
сосредоточиться на ректификации. Оценивание F с выбросами — отдельная задача.

In [ ]:
def fundamental(K1, K2, R, T):
    tx, ty, tz = np.asarray(T).ravel()
    cross = np.float64([[0, -tz, ty], [tz, 0, -tx], [-ty, tx, 0]])
    return np.linalg.inv(K2).T @ cross @ R @ np.linalg.inv(K1)


def epipolar_distances(points1, points2, F):
    lines = cv2.computeCorrespondEpilines(np.float64(points1).reshape(-1, 1, 2), 1, F).reshape(-1, 3)
    homogeneous = np.column_stack([points2, np.ones(len(points2))])
    return np.abs(np.sum(homogeneous * lines, axis=1)) / np.linalg.norm(lines[:, :2], axis=1)


def show_pair(left, right, points1=None, points2=None, title='Stereo pair'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    for ax, image, points in zip(axes, (left, right), (points1, points2)):
        ax.imshow(image, cmap='gray', vmin=0, vmax=255)
        for y in range(30, image.shape[0], 40):
            ax.axhline(y, color='cyan', alpha=.55, lw=.7)
        if points is not None:
            ax.scatter(points[:, 0], points[:, 1], c=np.arange(len(points)), cmap='turbo', s=15)
        ax.set(xlabel='u [px]', ylabel='v [px]')
    fig.suptitle(title)
    plt.show()

In [ ]:
scene = make_scene()
F = fundamental(scene['K'], scene['K'], scene['R'], scene['T'])
epi_px = epipolar_distances(scene['pts1'], scene['pts2'], F)
dy_before = np.abs(scene['pts1'][:, 1] - scene['pts2'][:, 1])
assert epi_px.max() < 1e-8 and np.median(dy_before) > 1
print('Raw median |dy| [px]:', np.median(dy_before))
print('Max point-to-epiline distance [px]:', epi_px.max())
show_pair(scene['left'], scene['right'], scene['pts1'], scene['pts2'], 'Before rectification: rows differ')

## TODO 1 · 15–30 минут. Ректификация (rectification)

Заполните 8 строк в `rectify_pair`: `stereoRectify` с `CALIB_ZERO_DISPARITY`,
`alpha=0`; карты для обеих камер через `initUndistortRectifyMap`, `CV_32FC1`;
затем `remap` с `INTER_LINEAR`. Верните изображения и геометрию в словаре.
`R1/R2` вращают лучи в новые камеры; `P1/P2` описывают их проекции;
`Q` переводит `(u,v,d,1)` в однородные 3D-координаты.

Проверим не только картинку: перенесём независимые соответствия через
`undistortPoints(..., R=..., P=...)`, после чего их строки должны совпасть.
GT уже относится к выходной сетке; варпить его исходной картой нельзя.

In [ ]:
def rectify_pair(left, right, K1, dist1, K2, dist2, R, T):
    if (left is None or right is None or left.ndim != 2 or left.shape != right.shape
            or left.dtype != np.uint8 or right.dtype != np.uint8):
        raise ValueError('Expected equal uint8 grayscale images')
    if not np.isfinite(T).all() or np.linalg.norm(T) <= 0:
        raise ValueError('Stereo baseline must be nonzero')
    size = (left.shape[1], left.shape[0])
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 8 строк: stereoRectify → две карты → remap, сохранить Q/P/R/ROI
    raise NotImplementedError("TODO 1")
    # ───────────────────────────────────────────────────────────

In [ ]:
rect = rectify_pair(scene['left'], scene['right'], scene['K'], scene['dist'],
                    scene['K'], scene['dist'], scene['R'], scene['T'])
p1 = cv2.undistortPoints(scene['pts1'].reshape(-1, 1, 2), scene['K'], scene['dist'],
                         R=rect['R1'], P=rect['P1']).reshape(-1, 2)
p2 = cv2.undistortPoints(scene['pts2'].reshape(-1, 1, 2), scene['K'], scene['dist'],
                         R=rect['R2'], P=rect['P2']).reshape(-1, 2)
dy_after = np.abs(p1[:, 1] - p2[:, 1])
assert dy_after.max() < 1e-8
assert rect['left'].shape == scene['gt'].shape and rect['Q'].shape == (4, 4)
print('Rectified max |dy| [px]:', dy_after.max())
show_pair(rect['left'], rect['right'], p1, p2, 'After rectification: search along each row')

## TODO 2 · 30–50 минут. SGBM и WLS

Стереосопоставление (stereo matching) ищет диспаратность `d=u_left-u_right`.
`num_disparities` кратно 16; `block_size` нечётно. Для серого изображения:
`P1=8*block_size**2`, `P2=32*block_size**2`, причём `P2>P1`.
`uniqueness` отсеивает неоднозначные совпадения; `speckle` задаёт размер
удаляемых малых компонент. Переберём эти четыре параметра ниже.

Заполните 12 строк: создайте левый и правый matcher, вычислите обе **знаковые
int16** карты; передайте их в WLS (`lambda=8000`, `sigmaColor=1.5`).
Только после фильтра поделите обе карты на **16** и замените `d<=0` на NaN.
Не берите модуль правой карты: её направление и знак задаёт `createRightMatcher`.
WLS использует левое изображение как направляющее и проверяет согласованность
левой/правой карт. Это постфильтр, он может заполнить ошибку и не гарантирует улучшения.

In [ ]:
def sgbm_wls(left, right, num_disparities=64, block_size=5, uniqueness=10, speckle=50):
    if (left is None or right is None or left.ndim != 2 or left.shape != right.shape
            or left.dtype != np.uint8 or right.dtype != np.uint8):
        raise ValueError('Expected equal uint8 grayscale images')
    if (not isinstance(num_disparities, int) or num_disparities <= 0 or num_disparities % 16
            or not isinstance(block_size, int) or block_size < 3 or block_size % 2 == 0
            or num_disparities + block_size >= left.shape[1] or uniqueness < 0 or speckle < 0):
        raise ValueError('Invalid SGBM parameters')
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 12 строк: raw int16 → right matcher → WLS → float pixels /16
    raise NotImplementedError("TODO 2")
    # ───────────────────────────────────────────────────────────

In [ ]:
matched = sgbm_wls(rect['left'], rect['right'])
assert matched['raw_left'].dtype == matched['raw_right'].dtype == np.int16
assert matched['disparity'].shape == matched['wls'].shape == scene['gt'].shape
assert np.nanmedian(matched['disparity'][scene['valid']]) < 40  # pixels, not fixed-point /16 codes
fig, axes = plt.subplots(1, 4, figsize=(15, 3), constrained_layout=True)
for ax, array, title in zip(axes, [scene['gt'], matched['disparity'], matched['wls'], matched['confidence']],
                            ['GT [px]', 'SGBM [px]', 'WLS [px]', 'WLS confidence [0..255]']):
    im = ax.imshow(array, cmap='turbo', vmin=0, vmax=255 if 'confidence' in title else 30)
    ax.set_title(title); fig.colorbar(im, ax=ax, shrink=.7)
plt.show()

## TODO 3 · 50–65 минут. Измерение качества

Заполните 10 строк `disparity_report`. Маска оценки определяется **GT**:
геометрическая видимость в обеих камерах, конечная положительная GT-диспаратность
и фиксированная область вдали от внешних границ. Левая полоса 80 px исключена
заранее для всех конфигураций (диапазон поиска до 64 px); границы объектов остаются.

`bad2_pct` — процент ошибок **строго >2 px** на этой фиксированной маске.
NaN/Inf/неположительный prediction на валидном GT считается ошибкой.
`coverage_pct` — доля валидных предсказаний на той же области.
Полный `rmse_px` возвращает `None`, если есть пропуски; условный
`rmse_valid_px` полезен для диагностики, но не заменяет полный RMSE.
Пустая GT-маска — ошибка. Эти соглашения совпадают с HW4.

In [ ]:
def disparity_report(prediction, truth, mask):
    prediction, truth, mask = np.asarray(prediction), np.asarray(truth), np.asarray(mask, dtype=bool)
    if prediction.shape != truth.shape or mask.shape != truth.shape:
        raise ValueError('Prediction, truth and mask shapes must match')
    support = mask & np.isfinite(truth) & (truth > 0)
    if not support.any():
        raise ValueError('Empty valid GT support')
    # ───────────────────────── ВАШ КОД ─────────────────────────
    # 10 строк: фиксированный знаменатель, invalid=bad, полный RMSE или None
    raise NotImplementedError("TODO 3")
    # ───────────────────────────────────────────────────────────

In [ ]:
boundary = disparity_report(np.array([8., 10., 11., np.nan]), np.full(4, 8.), np.ones(4, bool))
assert boundary['bad2_pct'] == 50 and boundary['coverage_pct'] == 75 and boundary['rmse_px'] is None
reports = {name: disparity_report(matched[name], scene['gt'], scene['valid']) for name in ('disparity', 'wls')}
print(json.dumps(reports, indent=2))
assert reports['disparity']['coverage_pct'] > 95 and reports['disparity']['bad2_pct'] < 10

## 65–80 минут. Сборка: тюнинг, глубина, облако

Каркас сетки готов: два значения каждого из четырёх параметров, 16 запусков.
Во всех строках один GT-знаменатель. Выбор по минимуму bad-2.0 на этом учебном
наборе — **тюнинг на GT**, не независимая оценка обобщения. В HW4 сохраните
конфигурацию и проверьте её без подстройки на других парах.
Сравните SGBM и WLS; меньшая шероховатость карты не гарантирует меньшей ошибки.

In [ ]:
grid = []
for nd, bs, uq, sp in itertools.product((32, 64), (3, 5), (5, 15), (0, 50)):
    result = sgbm_wls(rect['left'], rect['right'], nd, bs, uq, sp)
    for method in ('disparity', 'wls'):
        report = disparity_report(result[method], scene['gt'], scene['valid'])
        grid.append(dict(num_disparities=nd, block_size=bs, uniqueness=uq, speckle=sp, method=method, **report))
best = min((row for row in grid if row['method'] == 'disparity'), key=lambda row: row['bad2_pct'])
print('nd bs uq sp method bad2[%] coverage[%] RMSE[px]')
for row in grid:
    print(row['num_disparities'], row['block_size'], row['uniqueness'], row['speckle'], row['method'],
          f"{row['bad2_pct']:.4f}", f"{row['coverage_pct']:.4f}", row['rmse_px'])
print('Best SGBM on this GT:', best)
assert len(grid) == 32 and len({row['valid_pixels'] for row in grid}) == 1

In [ ]:
def stereo_depth(disparity, focal_px, baseline_m, doffs=0.):
    if focal_px <= 0 or baseline_m <= 0 or not np.isfinite([focal_px, baseline_m, doffs]).all():
        raise ValueError('Expected positive focal length and baseline')
    disparity = np.asarray(disparity, dtype=np.float64)
    good = np.isfinite(disparity) & (disparity > 0) & (disparity + doffs > 0)
    depth = np.full(disparity.shape, np.nan)
    np.divide(focal_px * baseline_m, disparity + doffs, out=depth, where=good)
    return depth


def cloud_from_disparity(disparity, Q):
    good = np.isfinite(disparity) & (disparity > 0)
    xyz = cv2.reprojectImageTo3D(np.where(good, disparity, 0).astype(np.float32), Q)
    xyz[~good | ~np.isfinite(xyz).all(axis=2) | (xyz[..., 2] <= 0)] = np.nan
    return xyz


def write_ply(path, xyz, image, mask=None):
    good = np.isfinite(xyz).all(axis=2) & (xyz[..., 2] > 0)
    if mask is not None:
        good &= mask
    points = xyz[good]
    colors = np.repeat(image[good, None], 3, axis=1) if image.ndim == 2 else image[good, ::-1]
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    header = ('ply\nformat ascii 1.0\nelement vertex ' + str(len(points))
              + '\nproperty float x\nproperty float y\nproperty float z'
              + '\nproperty uchar red\nproperty uchar green\nproperty uchar blue\nend_header\n')
    with path.open('w', encoding='utf-8', newline='\n') as stream:
        stream.write(header)
        np.savetxt(stream, np.column_stack([points, colors]), fmt='%.6f %.6f %.6f %d %d %d')
    return len(points)

Глубина `Z=fB/(d+doffs)` — осевая координата в **ректифицированной левой камере**.
Это не расстояние `norm(XYZ)` и не исходная мировая Z после поворота.
Здесь `CALIB_ZERO_DISPARITY` даёт `doffs=0`; в общем случае
`doffs=cx_right-cx_left`. `f` берём из **P1 после ректификации**, B — в метрах.
`reprojectImageTo3D` получает float-диспаратность в пикселях и Q от той же ректификации.
При сравнении с GT сохраняем фиксированную маску и отдельно сообщаем покрытие.

In [ ]:
focal = rect['P1'][0, 0]
baseline = np.linalg.norm(scene['T'])
doffs = rect['P2'][0, 2] - rect['P1'][0, 2]
depth = stereo_depth(matched['disparity'], focal, baseline, doffs)
xyz = cloud_from_disparity(matched['disparity'], rect['Q'])
ok = scene['valid'] & np.isfinite(depth)
np.testing.assert_allclose(xyz[..., 2][ok], depth[ok], rtol=2e-6)
gt_xyz = cloud_from_disparity(scene['gt'], rect['Q'])
np.testing.assert_allclose(gt_xyz[..., 2], scene['depth_gt'], rtol=2e-6)
depth_rmse = float(np.sqrt(np.mean((depth[ok] - scene['depth_gt'][ok])**2)))
count = write_ply(LAB_DIR / 'out/stereo.ply', xyz, rect['left'])
print('f [px], B [m], doffs [px]:', focal, baseline, doffs)
print('Depth conditional RMSE [m], GT coverage [%]:', depth_rmse, 100 * ok.sum() / scene['valid'].sum())
print('Exported finite positive-depth vertices:', count)
fig = plt.figure(figsize=(11, 4), constrained_layout=True)
ax = fig.add_subplot(121)
im = ax.imshow(depth, cmap='viridis', vmin=1, vmax=5); fig.colorbar(im, ax=ax, label='Z [m]')
ax.set_title('Depth in rectified left camera')
ax = fig.add_subplot(122, projection='3d')
sample = xyz[::8, ::8].reshape(-1, 3)
sample = sample[np.isfinite(sample).all(axis=1)]
ax.scatter(*sample.T, c=sample[:, 2], cmap='viridis', s=2)
ax.set(xlabel='X [m]', ylabel='Y [m]', zlabel='Z [m]')
plt.show()

Малая ошибка диспаратности: `abs(DeltaZ) ≈ Z**2 * abs(DeltaD)/(f*B)`.
Это локальное приближение: проверим точную разность для `DeltaD=0.25 px`
на нескольких расстояниях, затем сравним положительную и отрицательную ошибку.
На дальних объектах метрическая ошибка растёт, хотя ошибка в пикселях та же.

In [ ]:
distance = np.array([1., 2., 4., 8., 12.])
delta_d = .25
ideal_d = focal * baseline / distance
approx_error = distance**2 * delta_d / (focal * baseline)
exact_plus = np.abs(stereo_depth(ideal_d + delta_d, focal, baseline) - distance)
exact_minus = np.abs(stereo_depth(ideal_d - delta_d, focal, baseline) - distance)
print('Z[m] approximate[m] exact(+d)[m] exact(-d)[m]')
print(np.column_stack([distance, approx_error, exact_plus, exact_minus]))
assert np.all(np.diff(exact_plus) > 0) and np.all(exact_minus > exact_plus)
plt.figure(figsize=(7, 3))
plt.plot(distance, approx_error, 'o-', label='local approximation')
plt.plot(distance, exact_plus, 's--', label='exact +0.25 px')
plt.plot(distance, exact_minus, '^--', label='exact -0.25 px')
plt.xlabel('Z [m]'); plt.ylabel('|Delta Z| [m]'); plt.legend(); plt.grid(); plt.show()

## ⭐ Свой SAD matcher

По желанию реализуйте `sad_match(left, right, max_disparity=32, block_size=5)`:
поиск только вдоль строки, сумма абсолютных разностей (sum of absolute differences,
SAD) в полном окне; результат — float-диспаратность левого кадра. Без `np.roll`:
оно переносит правый край к левому и создаёт ложные соответствия.
Недоступные окна исключайте до `argmin`; при равенстве выберите меньшую диспаратность.
Верните NaN там, где нет полного окна или выбран `d=0`.

Сравните с `StereoBM` и SGBM на **той же** GT-маске: bad-2.0, RMSE, покрытие.
Проверьте точный сдвиг на независимой случайной паре и поведение у границы.
Объясните ошибки возле окклюзий и почему большое окно смешивает две глубины.
⭐ не нужна для основного конвейера; готового решения в стартере нет.

## 80–90 минут. Мост к HW4

| Здесь | В `homeworks/hw4-calib3d/template/calib3d.py` |
|---|---|
| `rectify_pair`, проверка строк | Базовые три пары уже ректифицированы; для своей пары используйте K/dist/R/T и remap |
| `sgbm_wls(...)["disparity"]` | `sgbm`: параметры и результат в px; WLS — отдельный эксперимент |
| `disparity_report` | `disparity_metrics`: bad2_pct, coverage_pct, полный rmse_px или null |
| `stereo_depth`, `cloud_from_disparity` | `stereo_depth`, `point_cloud`: метры, doffs, валидность и .ply |
| График точности | Вывод чувствительности и подтверждение на нескольких расстояниях |

Откройте три открытых `sample_data/case*` HW4: это **другие** данные, их ожидаемые
результаты приведены в README HW4. Зафиксируйте конфигурацию для следующих пар.
Своя калибровка из lab11 связывает K/dist с измерениями на плоскости; F сама по себе
метрический масштаб не задаёт. Текущие правила и сроки — в README HW4.
Внешние Middlebury/KITTI остаются опциональным расширением, без больших загрузок
для этой лабораторной. Перед экспортом реальных данных проверьте единицы и GT-маску.

Для самопроверки: `python -m pytest tests -q` из папки лабораторной. Пока функции
не заполнены, их численные тесты отмечаются skip; обычные ошибки не скрываются.
После заполнения ожидаются все зелёные тесты. Покажите TA выровненные строки,
таблицу SGBM/WLS и облако; объясните, почему RMSE с пропусками нельзя улучшать маской.